# ARTI 308 – Lab 5 (Rewritten for BNPL Dataset)
## Repayment Status Classification


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)


In [ ]:
DATA_PATH = 'bnpl_dataset.csv'
df = pd.read_csv(DATA_PATH)
df.head(10)

In [ ]:
print('Shape:', df.shape)
print('\nMissing values per column:')
display(df.isna().sum().to_frame('missing_count').T)
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
target_col = 'Repayment_Status'
df[target_col].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=target_col, data=df)
plt.title('Repayment Status Distribution')
plt.xlabel('Repayment Status')
plt.ylabel('Count')
plt.show()

In [ ]:
df.dtypes

In [ ]:
df_fe = df.copy()

# Financial feature engineering
df_fe['Income_to_Purchase_Ratio'] = df_fe['Annual_Income'] / (df_fe['Purchase_Amount'] + 1)
df_fe['Credit_Score_Normalized'] = df_fe['Credit_Score'] / 850

# Age bins
df_fe['Age_Group'] = pd.cut(df_fe['Customer_Age'], 
                            bins=[18,30,45,60,100],
                            labels=['18-30','31-45','46-60','60+'])

df_fe[['Annual_Income','Purchase_Amount','Income_to_Purchase_Ratio']].head(10)

In [ ]:
X = df_fe.drop(columns=[target_col, 'Transaction_ID'])
y = df_fe[target_col]

categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
], remainder='passthrough')

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()